# Simulation Configuration

**This is the single place to control all simulation parameters.**

Edit the cells below, then run the entire notebook (`Kernel → Restart & Run All`) to write `simulation_config.json`.  
All other notebooks (`01_ship_generation.ipynb`, `02_simulation.ipynb`) load their parameters from that JSON file.

---
### Workflow
```
1. Edit parameters in this notebook
2. Run all cells  →  simulation_config.json is written
3. Run 01_ship_generation.ipynb  →  ships.parquet is written
4. Run 02_simulation.ipynb       →  simulation outputs written
```

---
## Cell 1 — Simulation Duration & Time

In [37]:
# Total number of simulation days
# Examples: 365 = 1 year, 730 = 2 years, 1825 = 5 years
SIMULATION_DAYS = 365

# Length of one simulation interval in days
# 1/24 = 1 hour per interval (recommended; finer resolution = slower)
INTERVAL_SIZE = 1 / 24

# Random seed for reproducibility (set to None to disable)
RANDOM_SEED = None

---
## Cell 2 — Data Paths
Paths are relative to the `part_4_new_simulation/` directory unless absolute.

In [38]:
# Maritime shipping network (any NetworkX graph in gpickle format)
# Nodes must have: source ('port'|'choke_point'|other), portname, country, lon, lat
# Edges must have: length (km)
NETWORK_FILE = '../part_3_network_extraction/network_outputs/network_calibrated.gpickle'

# Root data directory
DATA_DIR = '../data/'

# Directory containing weight_trade_matrix_all_transport_modes_HS{N}.csv files
TRADE_MATRICES_DIR = DATA_DIR + 'all_trade_matrices/'

# HS code → product name + ship type mapping
HS_CODES_MAPPING_FILE = DATA_DIR + 'hs_codes_mapping.json'

# USD → metric ton conversion factors per HS code
CONVERSION_FACTORS_FILE = (
    '../part_2_trade_volume_conversion/'
    'trade_volume_conversion_output/trade_volume_conversion_general.json'
)

# Real-world merchant fleet statistics (for ship size distributions)
MERCHANT_FLEET_FILE = DATA_DIR + 'merchant_fleet_data.csv'

# IMF port data (for port selection weights in ship generation)
IMF_PORT_DATA_FILE = DATA_DIR + 'port_data_imf.csv'

# BACI country codes (ISO3 → trade-data country name)
BACI_CODES_FILE = DATA_DIR + 'raw_trade_data/BACI_country_codes.csv'

# HS codes to simulate (list of 2-digit integers)
# HS 77 is reserved/unused in COMTRADE; HS 98-99 are non-standard national codes
# Standard chapters are 01–97 (excluding 77) = 96 codes
HS_CODES_LIST = [i for i in range(1, 98) if i != 77]

---
## Cell 3 — Ship Type Parameters

---
## Cell 3b — Port Selection Parameters

Controls how ships choose their origin and destination ports within a country.
Port scoring is multiplicative: `size_score^α × type_score^β × distance_penalty`.

| Parameter | Description |
|-----------|-------------|
| `PORT_WEIGHT_ALPHA` | Exponent on the trade-share (import/export) component |
| `PORT_WEIGHT_BETA` | Exponent on the vessel-type match component |
| `DISTANCE_PENALTY_SCALE` | λ in `exp(−λ × max(0, ratio−1))`. Higher = stronger preference for geographically efficient ports |

**Distance ratio:** `ratio = proposed_port_pair_distance / country_pair_optimal_distance`  
- ratio = 1.00 → factor = 1.00 (optimal — no penalty)  
- ratio = 1.10 → factor ≈ 0.91 at λ=0.09  
- ratio = 2.00 → factor ≈ 0.91 (gentle penalty at λ=0.09)

*Default values are the result of the optimizer in `02a_optimize_port_weights.ipynb`.*  
*`01_ship_generation.ipynb` re-runs the optimizer automatically and will override these values at runtime.*

In [ ]:
# Exponents on port trade-share and vessel-type-match scores in port selection.
# Scoring: size_score^alpha * type_score^beta * distance_penalty
# Values below are from the optimizer (02a_optimize_port_weights.ipynb).
# 01_ship_generation.ipynb re-runs the optimizer automatically.
PORT_WEIGHT_ALPHA = 0.5349
PORT_WEIGHT_BETA  = 0.8908

# Distance penalty scale (λ).
# Controls how strongly ports with sub-optimal routing distances are penalized.
DISTANCE_PENALTY_SCALE = 0.0942

In [40]:
# Cruising speed by ship type (km/h)
SHIP_SPEEDS = {
    'tanker':       25,
    'bulk carrier': 28,
    'cargo ship':   32,
}

# Average port loading time by ship type (days)
# Source: UNCTAD Review of Maritime Transport
PORT_LOADING_TIMES = {
    'tanker':       1.04,
    'bulk carrier': 2.13,
    'cargo ship':   0.71,
}

# Average port unloading time by ship type (days)
PORT_UNLOADING_TIMES = {
    'tanker':       1.04,
    'bulk carrier': 2.13,
    'cargo ship':   0.71,
}

# Gamma distribution: P(DWT < max_DWT) = CAPACITY_QUANTILE
CAPACITY_QUANTILE = 0.99

# Dirichlet concentration parameter for multi-commodity cargo ships
# Lower = more spread across HS codes; higher = more concentrated on dominant code
DIRICHLET_CONCENTRATION = 1

---
## Cell 4 — Port & Queuing Parameters

In [41]:
# Target utilization factor ρ = λ / (c × μ) for M/M/c queuing model
# Lower = less congestion, more berths allocated.  Typical range: 0.7 – 0.9
TARGET_RHO = 0.8

# Minimum number of berths per port (prevents 0-berth ports)
MIN_PORT_CAPACITY = 1

---
## Cell 5 — Choke Point Throughput

Controls the base throughput (ships per interval) at each choke point.

| Value | Behaviour |
|-------|----------|
| `null` / `None` | **Passthrough** — ships move freely at full capacity. Queueing only activates if an interruption event reduces the multiplier below 1.0 |
| Integer | Ships per interval that can pass at `capacity_multiplier = 1.0` |

In [42]:
# Choke points with THROUGHPUT-BASED queuing (ships/interval).
# Use None for passthrough (no queue unless an InterruptionEvent fires).
# DO NOT list canals here — they are configured in CANAL_CHOKEPOINTS below.
# Keys must match the 'name' attribute of choke_point nodes in the network.
CHOKE_POINT_THROUGHPUT = {
    # 'Strait of Gibraltar': None,
    # 'Bosphorus': 15,  # Example: ships/interval limit
}


### Cell 5b — Canal Chokepoints
Canals (Suez, Panama) use a **transit-slot model**: ships queue and then spend a fixed time transiting the canal.  Capacity is auto-calibrated before the simulation loop so that utilisation ρ ≈ `CANAL_TARGET_RHO` at baseline traffic.

In [ ]:
# Canal chokepoints: {canal_name: transit_time_hours}
# canal_name must exactly match the 'name' attribute of the choke_point node
# in the network.  transit_time_hours is the fixed time a ship spends
# actively transiting (excluding any wait-in-queue time).
CANAL_CHOKEPOINTS = {
    'Suez Canal':   14,   # ~14 h typical transit time
    'Panama Canal': 22,   # ~22 h total transit time in canal waters
}

# Target utilisation factor ρ for canal capacity calibration.
# Separate from TARGET_RHO (ports) to allow independent tuning.
# Lower = more transit slots (less congestion); typical range: 0.5 – 0.85
CANAL_TARGET_RHO = 0.7

# -----------------------------------------------------------------------
# Canal daily throughput caps (ships/day) — explicit capacity pre-assignment
# -----------------------------------------------------------------------
# When set, preassign_canal_capacities() runs before the simulation and
# randomly selects floor(CANAL_TARGET_RHO * daily_rate * SIMULATION_DAYS)
# ships to keep their canal route; the rest are rerouted around the canal.
# Ships that cannot find an alternative path keep their original route.
# Leave empty ({}) to disable pre-assignment and rely solely on the
# transit-slot queuing model.
#
# Effective annual throughput (at CANAL_TARGET_RHO=0.7, 365 days):
#   Suez Canal:   floor(0.7 × 38 × 365)  =  9,709 ships  (~75% of historical 50/day)
#   Panama Canal: floor(0.7 × 17 × 365)  =  4,343 ships  (~50% of historical 34/day)
CANAL_DAILY_RATES = {
    'Suez Canal':   38,   # ~38 ships/day → ~75% of baseline (~4M→~3M tons)
    'Panama Canal': 17,   # ~17 ships/day → ~50% of baseline (~2M→~1M tons)
}

# -----------------------------------------------------------------------
# Canal DWT-band restrictions (optional per-canal DWT filters)
# -----------------------------------------------------------------------
# exclude_above_dwt: ships with DWT >= this value are always rerouted.
# groups: list of DWT bands, each with an independent daily_rate cap.
#   Ships not covered by any group (e.g., oversized) are already excluded
#   by exclude_above_dwt.  The min_dwt/max_dwt ranges should be contiguous
#   and together cover [0, exclude_above_dwt).
#
# Panama group rates set to halve historical throughput per band:
#   Panamax:    floor(0.7 × 13 × 365) = 3,321  (~52% of historical 25/day)
#   Neopanamax: floor(0.7 ×  4 × 365) = 1,022  (~44% of historical  9/day)
#   Combined:   4,343 / 8,686         = 50.0%
CANAL_DWT_RESTRICTIONS = {
    'Panama Canal': {
        'exclude_above_dwt': 120_000,   # Neo-Panamax max draught limit
        'groups': [
            {
                'label':      'panamax',
                'min_dwt':    0,
                'max_dwt':    52_500,
                'daily_rate': 13,        # Halved from 25: ~13 ships/day
            },
            {
                'label':      'neopanamax',
                'min_dwt':    52_500,
                'max_dwt':    120_000,
                'daily_rate': 4,         # Halved from 9: ~4 ships/day
            },
        ],
    },
}

---
## Cell 6 — Rerouting Parameters

In [44]:
# How many alternative routes to evaluate when a ship considers rerouting
K_ALTERNATIVE_ROUTES = 3

# Patience multiplier for the rerouting decision rule:
#   Reroute if:  best_alt_travel_time  <  expected_remaining_current  ×  multiplier
#   1.0 = reroute as soon as any alternative is faster
#   1.2 = reroute only if alternative is 20% faster (adds tolerance for waiting)
REROUTE_PATIENCE_MULTIPLIER = 1.0

# Proactive rerouting: when True, traveling ships scan ahead for congested
# choke points and reroute BEFORE reaching them if an alternative is faster.
# When False, ships only reroute reactively (when already blocked at a node).
PROACTIVE_REROUTING = True

---
## Cell 7 — Physical Interruption Events

Each event specifies a disruption to a port or choke point.

| Field | Type | Description |
|-------|------|-------------|
| `day` | float | Simulation day when disruption starts |
| `end_day` | float or `None` | Simulation day when capacity is restored. `None` = permanent |
| `type` | str | `'port'` or `'choke_point'` |
| `target` | str | Port name or choke point name (must match network) |
| `capacity_multiplier` | float | `0.0` = full closure, `0.5` = 50% capacity, `1.0` = no change |

Leave as an empty list `[]` for no interruptions.

In [45]:
INTERRUPTION_EVENTS = [
    # --- Examples (uncomment to activate) ---

    # Full Suez Canal closure for 30 days starting day 60
    # {
    #     'day': 60,
    #     'end_day': 90,
    #     'type': 'choke_point',
    #     'target': 'Suez Canal',
    #     'capacity_multiplier': 0.0,
    # },

    # Port Said reduced to 30% capacity for 2 weeks
    # {
    #     'day': 100,
    #     'end_day': 114,
    #     'type': 'port',
    #     'target': 'Port Said',
    #     'capacity_multiplier': 0.3,
    # },

    # Permanent closure of a port (no end_day)
    # {
    #     'day': 200,
    #     'end_day': None,
    #     'type': 'port',
    #     'target': 'Larnaca',
    #     'capacity_multiplier': 0.0,
    # },
]

---
## Cell 8 — Economic Adjustment Events

Adjusts trade volumes for specific country/commodity combinations.

| Field | Type | Description |
|-------|------|-------------|
| `day` | float | `0` = baseline (applied before ship generation). `>0` = mid-simulation event |
| `country` | str | Country name (must match network / trade matrix index) |
| `direction` | str | `'export'`, `'import'`, or `'both'` |
| `hs_codes` | list | HS code integers to adjust. `[]` = all HS codes |
| `adjustment_pct` | float | Percentage change. `-10` = 10% reduction. `+20` = 20% increase |

Multiple events for the same country/HS code are **cumulative** within each epoch.

Leave as an empty list `[]` for no economic adjustments.

In [46]:
ECONOMIC_EVENTS = [
    # --- Examples (uncomment to activate) ---

    # Baseline: Russia's oil exports reduced 30% (sanctions)
    # {
    #     'day': 0,
    #     'country': 'Russian Federation',
    #     'direction': 'export',
    #     'hs_codes': [27],
    #     'adjustment_pct': -30,
    # },

    # Mid-simulation: further 10% reduction from day 180
    # {
    #     'day': 180,
    #     'country': 'Russian Federation',
    #     'direction': 'export',
    #     'hs_codes': [27],
    #     'adjustment_pct': -10,
    # },

    # Baseline: Egypt's agricultural exports increase 20%
    # {
    #     'day': 0,
    #     'country': 'Egypt',
    #     'direction': 'export',
    #     'hs_codes': [7, 8, 10],
    #     'adjustment_pct': 20,
    # },
]

---
## Cell 9 — Output & Checkpoint Settings

In [ ]:
# Directory for all simulation outputs (relative to part_4_new_simulation/)
# -----------------------------------------------------------------------
# SCENARIO CONTROL: change this to switch between scenarios.
# All four pipeline notebooks (00–02) write to / read from this same directory.
# The create_scenario_config.py script overrides this automatically per scenario.
# -----------------------------------------------------------------------
OUTPUT_DIR = 'simulation_output_data/scenario_baseline/'

# Save a checkpoint every N simulation days (0 = disabled)
# Checkpoints allow resuming a run after interruption
CHECKPOINT_INTERVAL_DAYS = 30

# Whether to record ship positions each timestep
# Large runs: set LOCATION_SAMPLE_INTERVAL > 1 to reduce output size
SAVE_SHIP_LOCATIONS = True
LOCATION_SAMPLE_INTERVAL = 1   # Save every N timesteps (1 = every hour)

# Also write CSV copies of key outputs for backward compatibility
# with part_5_visualization notebooks
BACKWARD_COMPAT_CSV = True

---
## Cell 10 — Serialize to simulation_config.json

Run this cell (or the entire notebook) to write the config file.  
**Do not edit `simulation_config.json` directly** — changes will be overwritten.

In [ ]:
import json
from pathlib import Path

config = {
    # Cell 1
    'SIMULATION_DAYS':              SIMULATION_DAYS,
    'INTERVAL_SIZE':                INTERVAL_SIZE,
    'RANDOM_SEED':                  RANDOM_SEED,
    # Cell 2
    'NETWORK_FILE':                 NETWORK_FILE,
    'DATA_DIR':                     DATA_DIR,
    'TRADE_MATRICES_DIR':           TRADE_MATRICES_DIR,
    'HS_CODES_MAPPING_FILE':        HS_CODES_MAPPING_FILE,
    'CONVERSION_FACTORS_FILE':      CONVERSION_FACTORS_FILE,
    'MERCHANT_FLEET_FILE':          MERCHANT_FLEET_FILE,
    'IMF_PORT_DATA_FILE':           IMF_PORT_DATA_FILE,
    'BACI_CODES_FILE':              BACI_CODES_FILE,
    'HS_CODES_LIST':                HS_CODES_LIST,
    # Cell 3
    'SHIP_SPEEDS':                  SHIP_SPEEDS,
    'PORT_LOADING_TIMES':           PORT_LOADING_TIMES,
    'PORT_UNLOADING_TIMES':         PORT_UNLOADING_TIMES,
    'CAPACITY_QUANTILE':            CAPACITY_QUANTILE,
    'DIRICHLET_CONCENTRATION':      DIRICHLET_CONCENTRATION,
    # Cell 3b
    'PORT_WEIGHT_ALPHA':            PORT_WEIGHT_ALPHA,
    'PORT_WEIGHT_BETA':             PORT_WEIGHT_BETA,
    'DISTANCE_PENALTY_SCALE':       DISTANCE_PENALTY_SCALE,
    # Cell 4
    'TARGET_RHO':                   TARGET_RHO,
    'MIN_PORT_CAPACITY':            MIN_PORT_CAPACITY,
    # Cell 5
    'CHOKE_POINT_THROUGHPUT':       CHOKE_POINT_THROUGHPUT,
    # Cell 5b
    'CANAL_CHOKEPOINTS':            CANAL_CHOKEPOINTS,
    'CANAL_TARGET_RHO':             CANAL_TARGET_RHO,
    'CANAL_DAILY_RATES':            CANAL_DAILY_RATES,
    'CANAL_DWT_RESTRICTIONS':       CANAL_DWT_RESTRICTIONS,
    # Cell 6
    'K_ALTERNATIVE_ROUTES':         K_ALTERNATIVE_ROUTES,
    'REROUTE_PATIENCE_MULTIPLIER':  REROUTE_PATIENCE_MULTIPLIER,
    'PROACTIVE_REROUTING':          PROACTIVE_REROUTING,
    # Cell 7
    'INTERRUPTION_EVENTS':          INTERRUPTION_EVENTS,
    # Cell 8
    'ECONOMIC_EVENTS':              ECONOMIC_EVENTS,
    # Cell 9
    'OUTPUT_DIR':                   OUTPUT_DIR,
    'CHECKPOINT_INTERVAL_DAYS':     CHECKPOINT_INTERVAL_DAYS,
    'SAVE_SHIP_LOCATIONS':          SAVE_SHIP_LOCATIONS,
    'LOCATION_SAMPLE_INTERVAL':     LOCATION_SAMPLE_INTERVAL,
    'BACKWARD_COMPAT_CSV':          BACKWARD_COMPAT_CSV,
}

out_path = Path('simulation_config.json')
with open(out_path, 'w') as f:
    json.dump(config, f, indent=2)

print(f'Config written to {out_path.resolve()}')
print(f'  Simulation: {SIMULATION_DAYS} days  |  interval: {INTERVAL_SIZE*24:.1f} h')
print(f'  HS codes: {len(HS_CODES_LIST)}')
print(f'  Port weight alpha/beta: {PORT_WEIGHT_ALPHA}/{PORT_WEIGHT_BETA}')
print(f'  Distance penalty scale: {DISTANCE_PENALTY_SCALE}')
print(f'  Canal daily rates: {CANAL_DAILY_RATES}')
print(f'  Interruption events: {len(INTERRUPTION_EVENTS)}')
print(f'  Economic events:     {len(ECONOMIC_EVENTS)}')
print(f'  Output dir:          {OUTPUT_DIR}')
print(f'  Checkpoint every:    {CHECKPOINT_INTERVAL_DAYS} days')
print()
print('Next step: run 00_precompute_routes.ipynb (if network changed), then 01_ship_generation.ipynb')